# 01 — Reading the raw data

**The claim being tested.** `ghs_adapter.py` says every persona carries the household's
*real* reported net monthly income in rand, "looked up, never modelled". That number
becomes the affordability tier the whole product economy rests on.

**What you are learning.** Survey microdata is not a spreadsheet. A number in a column
can be a value *or* a code meaning "refused to answer". Telling those apart is the job.

**Rule for this notebook:** write your guess down before you run each cell. Being wrong
on purpose is how it sticks.

In [ ]:
import sys
sys.path.insert(0, "D:/Fub-learn/learn")

import pandas as pd
import numpy as np
import pyreadstat
import paths

pd.set_option("display.max_columns", 60)
paths.check()

## 1. Open the file yourself

Not through the adapter. Raw.

In [ ]:
hh, hmeta = pyreadstat.read_dta(str(paths.GHS_HOUSEHOLD), encoding="WINDOWS-1252")

print(f"{len(hh):,} households, {hh.shape[1]} columns")
hh[["uqnr", "fin_reqinc"]].head()

The `encoding=` argument is not decoration. Stats SA writes value labels in
Windows-1252 (there are en-dashes in the fee bands) and pyreadstat crashes on byte
`0x96` without it. Try removing it once, so you have met the error.

## 2. Look at the income column before trusting it

**Guess first.** Write down: what do you expect the *highest* household income in
South Africa's GHS to be?

In [ ]:
inc = hh["fin_reqinc"]

print(inc.describe())
print("\nTop 10 values:")
print(inc.sort_values(ascending=False).head(10).to_string())

That top value is not a person. It is a **sentinel** — a code standing in for
"unspecified". Count them.

In [ ]:
SENTINEL = 9999999.0

n_sent = (inc == SENTINEL).sum()
print(f"sentinel rows : {n_sent}  ({n_sent / len(hh):.1%} of households)")

real = inc[inc != SENTINEL]
print(f"real max      : R{real.max():,.0f}")
print(f"real median   : R{real.median():,.0f}")
print(f"real mean     : R{real.mean():,.0f}")

## 3. Now feel the size of the mistake

What would the mean look like if nobody had caught the sentinel?

In [ ]:
naive = inc.mean()
clean = real.mean()

print(f"naive mean (sentinel left in) : R{naive:,.0f}")
print(f"clean mean                    : R{clean:,.0f}")
print(f"inflation factor              : {naive / clean:.1f}x")
print()
print("Note what does NOT happen: no error, no warning, no crash.")
print("The number is simply wrong, and everything downstream stays fluent.")

**This is the whole lesson of topic 01.** In an LLM system, a data bug does not
produce garbage output. It produces confident, well-written, wrong output. The only
defence is being able to open the file yourself.

## 4. Check the production code actually handles it

The adapter claims it does. Verify, don't assume.

In [ ]:
src = (paths.SCRIPTS / "ghs_adapter.py").read_text(encoding="utf-8")

for i, line in enumerate(src.splitlines(), 1):
    if "SENTINEL" in line or "9999999" in line:
        print(f"{i:5}  {line}")

## 5. Your turn — hunt for the ones nobody caught

`fin_reqinc` was handled. Other columns may not be. A sentinel usually looks like a
run of 9s, or an 88 / 99 / 98 sitting far above the real range.

In [ ]:
def sentinel_smell(df, max_report=25):
    """Flag numeric columns whose max is suspiciously detached from the rest."""
    rows = []
    for col in df.select_dtypes(include=[np.number]).columns:
        s = df[col].dropna()
        if s.empty or s.nunique() < 3:
            continue
        p99, mx = s.quantile(0.99), s.max()
        if p99 > 0 and mx > p99 * 5:
            rows.append({
                "column": col,
                "p99": p99,
                "max": mx,
                "gap": mx / p99,
                "n_at_max": int((s == mx).sum()),
            })
    out = pd.DataFrame(rows).sort_values("gap", ascending=False)
    return out.head(max_report)


sentinel_smell(hh)

For any column that looks suspicious, read its real meaning from the file's own
metadata rather than guessing:

In [ ]:
COL = "fin_reqinc"   # <- change this to a column from the table above

print("label :", hmeta.column_names_to_labels.get(COL))
print("values:", hmeta.variable_value_labels.get(COL, "(no value labels — free numeric)"))

## 6. Missingness

A sentinel is a loud missing value. A blank is a quiet one. Both matter, because a
field that is empty 40% of the time cannot carry a persona.

In [ ]:
used_by_adapter = [
    "uqnr", "fin_reqinc", "com_int_fixed", "com_int_mobile", "hwl_assets_comp",
]

present = [c for c in used_by_adapter if c in hh.columns]
miss = hh[present].isna().mean().sort_values(ascending=False)
miss.to_frame("share missing").style.format("{:.1%}")

## 7. The real exercise: rebuild one field by hand

Pick one thing the adapter produces — the income band, say — and compute it yourself
straight from the raw column. Then diff your answer against the adapter's.

Every row where you disagree is either a bug in the adapter or a gap in your
understanding. Both are worth finding, and you will not know which until you look.

In [ ]:
sys.path.insert(0, str(paths.SCRIPTS))

# import ghs_adapter as ghs        # the production code, from main
# your turn.

---

### Before you move on

You should now be able to say, without opening a file:

- how many GHS households report an income at all;
- what the real top income in the survey is;
- what would happen to your product if one sentinel slipped through.

Next: **02 — uncertainty**, where you find out how much a 30-person panel can
actually tell you.